# Spatial join
Example: assign neighborhood to each restaurant


In [2]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as cx

nyc = gpd.read_file("./data/nyc.geojson")
nyc_restaurant = pd.read_csv("./data/google_maps_restaurants(cleaned).csv")


In [4]:
# convert nyc_restaurant to geopandas dataframe, specify CRS the same as nyc.crs
nyc_restaurant_gdf = gpd.GeoDataFrame(
    nyc_restaurant, geometry=gpd.points_from_xy(nyc_restaurant.Lon, nyc_restaurant.Lat), crs=nyc.crs)

In [6]:
# inspect crs

nyc_restaurant_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [11]:
# use sjoin to identify which neighborhood (Community District) each restaurant is in

joined_gdf = nyc_restaurant_gdf.sjoin(nyc, how='inner', predicate='intersects')

joined_gdf[['Name','Rating','Rating Count','Price Category',"Address",'BoroName','NTAName','CDTANAME']].head()

,Name,Rating,Rating Count,Price Category,Address,BoroName,NTAName,CDTANAME
2,Joe's Restaurant,4.6,812.0,2.0,"6611 Forest Ave, Ridgewood, NY 11385",Queens,Ridgewood,QN05 Ridgewood-Maspeth-Middle Village (CD 5 Ap...
6,Mexican Cantina Restaurant & Bar,4.5,257.0,2.0,"1736 Victory Blvd, Staten Island, NY 10314",Staten Island,Westerleigh-Castleton Corners,SI01 North Shore (CD 1 Equivalent)
8,Golden Krust Caribbean Restaurant,3.7,82.0,1.0,"1014a Nostrand Ave, Brooklyn, NY 11225",Brooklyn,Prospect Lefferts Gardens-Wingate,BK09 Crown Heights (South) (CD 9 Approximation)
12,POLONICA RESTAURANT,4.5,227.0,2.0,"8303 3rd Ave, Brooklyn, NY 11209",Brooklyn,Bay Ridge,BK10 Bay Ridge-Dyker Heights (CD 10 Approximat...
13,Serafina Italian Restaurant Tribeca,4.3,1057.0,2.0,"95 W Broadway, New York, NY 10007",Manhattan,Tribeca-Civic Center,MN01 Financial District-Tribeca (CD 1 Equivalent)


In [19]:
# Which community has the most expensive

joined_gdf.groupby("CDTANAME", as_index=False).agg(
    Boro=("BoroName", "first"),
    NTAName=("NTAName", "first"),
    count=('Rating', 'count'),
    avg_rating=('Rating', 'mean'),
    avg_price_category=('Price Category', 'mean'),
)

,CDTANAME,Boro,NTAName,count,avg_rating,avg_price_category
0,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),Brooklyn,Greenpoint,10,4.260000,1.750000
1,BK02 Downtown Brooklyn-Fort Greene (CD 2 Appro...,Brooklyn,Brooklyn Heights,17,4.300000,1.941176
2,BK03 Bedford-Stuyvesant (CD 3 Approximation),Brooklyn,Bedford-Stuyvesant (East),2,4.350000,1.000000
3,BK04 Bushwick (CD 4 Equivalent),Brooklyn,Bushwick (West),8,4.237500,1.571429
4,BK05 East New York-Cypress Hills (CD 5 Approxi...,Brooklyn,Cypress Hills,10,3.860000,1.333333
5,BK06 Park Slope-Carroll Gardens (CD 6 Approxim...,Brooklyn,Carroll Gardens-Cobble Hill-Gowanus-Red Hook,10,4.430000,1.555556
6,BK07 Sunset Park-Windsor Terrace (CD 7 Approxi...,Brooklyn,Sunset Park (West),12,4.075000,1.500000
7,BK08 Crown Heights (North) (CD 8 Approximation),Brooklyn,Crown Heights (North),5,4.140000,1.750000
8,BK09 Crown Heights (South) (CD 9 Approximation),Brooklyn,Prospect Lefferts Gardens-Wingate,8,4.100000,1.285714
9,BK10 Bay Ridge-Dyker Heights (CD 10 Approximat...,Brooklyn,Bay Ridge,18,4.477778,1.764706
